In [5]:
import pandas as pd
import os 
import warnings
warnings.filterwarnings("ignore")

# Load main table
app_train = pd.read_csv('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/application_train.csv')
print(f"app_train: {app_train.shape}")

# Load aggregated tables
prev_full = pd.read_csv('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/processed/prev_app_full_final.csv')
print(f"prev_full: {prev_full.shape}")

bureau_agg = pd.read_csv('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/processed/bureau_agg_final.csv')
print(f"bureau_agg: {bureau_agg.shape}")

app_train: (307511, 122)
prev_full: (338857, 58)
bureau_agg: (305811, 22)


In [6]:
# Step 1 — merge app_train with bureau_agg
df = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(f"After bureau merge: {df.shape}")

# Step 2 — merge with prev_full
df = df.merge(prev_full, on='SK_ID_CURR', how='left')
print(f"After prev_full merge: {df.shape}")

After bureau merge: (307511, 143)
After prev_full merge: (307511, 200)


In [7]:
# Debt burden ratios — combine bureau debt with main table income
df['debt_to_income'] = df['bur_total_debt'] / df['AMT_INCOME_TOTAL'].replace(0, None)
df['annuity_to_income'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'].replace(0, None)
df['credit_to_income'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'].replace(0, None)

# How current application compares to previous behaviour
df['prev_credit_to_current'] = df['prev_avg_credit'] / df['AMT_CREDIT'].replace(0, None)

# External debt vs current application
df['bur_debt_to_current_credit'] = df['bur_total_debt'] / df['AMT_CREDIT'].replace(0, None)

# Late payment burden weighted by debt
df['late_payment_debt_burden'] = df['inst_late_payment_rate'] * df['bur_total_debt']

# Clean up infinities
df = df.replace([float('inf'), float('-inf')], None)

print(df.shape)
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False).head(20)

(307511, 206)


COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_AVG     213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_AVG               208642
FLOORSMIN_MEDI              208642
YEARS_BUILD_MODE            204488
YEARS_BUILD_AVG             204488
YEARS_BUILD_MEDI            204488
OWN_CAR_AGE                 202929
LANDAREA_AVG                182590
LANDAREA_MEDI               182590
LANDAREA_MODE               182590
dtype: int64

In [8]:
new_features = ['debt_to_income', 'annuity_to_income', 'credit_to_income', 
                'prev_credit_to_current', 'bur_debt_to_current_credit', 
                'late_payment_debt_burden']

df[new_features].isnull().sum()

debt_to_income                44020
annuity_to_income                12
credit_to_income                  0
prev_credit_to_current        16454
bur_debt_to_current_credit    44020
late_payment_debt_burden      58004
dtype: int64

In [9]:
print(df.shape)

(307511, 206)


## Revising the Prev_App_Combined with some new features

In [10]:
import duckdb as db
con = db.connect()

# --- POS Cash: 3 and 6 month windows ---
con.execute("""
    CREATE VIEW pos_3m AS
    SELECT
        SK_ID_CURR,
        AVG(SK_DPD)                                                    as pos_avg_dpd_3m,
        MAX(SK_DPD)                                                    as pos_max_dpd_3m,
        SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)                  as pos_late_count_3m,
        ROUND(SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) 
            * 1.0 / COUNT(*), 4)                                      as pos_late_rate_3m
    FROM read_csv_auto('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/POS_CASH_balance.csv')
    WHERE MONTHS_BALANCE >= -3
    GROUP BY SK_ID_CURR
""")

con.execute("""
    CREATE VIEW pos_6m AS
    SELECT
        SK_ID_CURR,
        AVG(SK_DPD)                                                    as pos_avg_dpd_6m,
        MAX(SK_DPD)                                                    as pos_max_dpd_6m,
        SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)                  as pos_late_count_6m,
        ROUND(SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) 
            * 1.0 / COUNT(*), 4)                                      as pos_late_rate_6m
    FROM read_csv_auto('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/POS_CASH_balance.csv')
    WHERE MONTHS_BALANCE >= -6
    GROUP BY SK_ID_CURR
""")

# --- Credit Card: 3 and 6 month windows ---
con.execute("""
    CREATE VIEW cc_3m AS
    SELECT
        SK_ID_CURR,
        AVG(AMT_BALANCE)                                               as cc_avg_balance_3m,
        AVG(CASE WHEN AMT_CREDIT_LIMIT_ACTUAL > 0 
            THEN AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL 
            ELSE NULL END)                                             as cc_avg_utilization_3m,
        AVG(SK_DPD)                                                    as cc_avg_dpd_3m,
        MAX(SK_DPD)                                                    as cc_max_dpd_3m,
        SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)                  as cc_late_count_3m,
        ROUND(SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) 
            * 1.0 / COUNT(*), 4)                                      as cc_late_rate_3m
    FROM read_csv_auto('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/credit_card_balance.csv')
    WHERE MONTHS_BALANCE >= -3
    GROUP BY SK_ID_CURR
""")

con.execute("""
    CREATE VIEW cc_6m AS
    SELECT
        SK_ID_CURR,
        AVG(AMT_BALANCE)                                               as cc_avg_balance_6m,
        AVG(CASE WHEN AMT_CREDIT_LIMIT_ACTUAL > 0 
            THEN AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL 
            ELSE NULL END)                                             as cc_avg_utilization_6m,
        AVG(SK_DPD)                                                    as cc_avg_dpd_6m,
        MAX(SK_DPD)                                                    as cc_max_dpd_6m,
        SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)                  as cc_late_count_6m,
        ROUND(SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) 
            * 1.0 / COUNT(*), 4)                                      as cc_late_rate_6m
    FROM read_csv_auto('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/credit_card_balance.csv')
    WHERE MONTHS_BALANCE >= -6
    GROUP BY SK_ID_CURR
""")

# --- Installments: recent 1 year window ---
con.execute("""
    CREATE VIEW inst_1y AS
    SELECT
        SK_ID_CURR,
        AVG(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)                     as inst_avg_delay_1y,
        MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)                     as inst_max_delay_1y,
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT 
            THEN 1 ELSE 0 END)                                        as inst_late_count_1y,
        ROUND(SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT 
            THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4)                  as inst_late_rate_1y
    FROM read_csv_auto('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/raw/installments_payments.csv')
    WHERE DAYS_INSTALMENT >= -365
    GROUP BY SK_ID_CURR
""")

print("All time-windowed views created")

All time-windowed views created


In [11]:
# Fetch all windowed features
pos_3m = con.execute("SELECT * FROM pos_3m").df()
pos_6m = con.execute("SELECT * FROM pos_6m").df()
cc_3m = con.execute("SELECT * FROM cc_3m").df()
cc_6m = con.execute("SELECT * FROM cc_6m").df()
inst_1y = con.execute("SELECT * FROM inst_1y").df()

# Merge into main df
df = df.merge(pos_3m, on='SK_ID_CURR', how='left')
df = df.merge(pos_6m, on='SK_ID_CURR', how='left')
df = df.merge(cc_3m, on='SK_ID_CURR', how='left')
df = df.merge(cc_6m, on='SK_ID_CURR', how='left')
df = df.merge(inst_1y, on='SK_ID_CURR', how='left')

# Fill nulls — no recent history means 0 activity
windowed_cols = [c for c in df.columns if c.endswith(('_3m', '_6m', '_1y'))]
df[windowed_cols] = df[windowed_cols].fillna(0)

print(f"Shape after windowed features: {df.shape}")

Shape after windowed features: (307511, 230)


## Reducing the float size

In [12]:
# Downcast all float64 columns to float32
float_cols = df.select_dtypes(include='float64').columns
df[float_cols] = df[float_cols].astype('float32')

# Downcast int64 to int32 where possible
int_cols = df.select_dtypes(include='int64').columns
df[int_cols] = df[int_cols].astype('int32')

print(f"Memory usage before: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Memory usage before: 327.6 MB


In [13]:
import pyarrow

df.to_parquet(
    'C:/Users/BeratErcan/Home-Credit-Default-Risk/data/processed/final_merged.parquet',
    index=False,
    compression='snappy'  # fast compression, good balance of size vs speed
)

size = os.path.getsize('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/processed/final_merged.parquet') / (1024*1024)
print(f"Parquet size: {size:.1f} MB")

Parquet size: 61.4 MB


In [14]:
df_check = pd.read_parquet('C:/Users/BeratErcan/Home-Credit-Default-Risk/data/processed/final_merged.parquet')
print(df_check.shape)
print(df_check.dtypes.value_counts())

(307511, 230)
float32    173
int32       41
str         16
Name: count, dtype: int64
